In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Hypothesis

Our general hypothesis is that the location of plants are disproportionately placed in marginalized communities. The characteristic of a marginalized community is not singular, which illicts the need to conduct multiple hypothesis testing to observe potential inequalities across multiple socioeconomic demographic metrics.

Our alternative hypotheses are as follows:
1) Plants are disproportionately located in areas with higher % People of Color
2) Plants are disproportionately located in areas with higher % Low Income
3) Plants are disproportionately located in areas with higher % Less Than High School Education
4) Plants are disproportionately located in areas with higher % Limited English speaking households
5) Plants are disproportionately located in areas with higher % Unemployment Rate
6) There is a difference in the percentage of people under age 5 for counties with plants vs. without
7) There is a difference in the percentage of people over age 64 for counties with plants vs. without


Our null hypotheses are that there is no difference in means of these demographic factors between counties with plants and counties without plants.

We will be conducting A/B testing against each hypothesis. This is because we can treat each plant in the eGRID data as the 'treatment' of having a plant. The other counties across the US will be the control groups, for not having a plant. 

To correct for the multiple hypothesis tests, we will use two different methods:
- To control the FDR at 0.05, we will use the Benjamini–Yekutieli procedure, which controls the false discovery rate under arbitrary dependence assumptions. This is needed because the demographic metrics are not independent due to the socioeconomic functioning of the US.(could add EDA on this w a correlation map)
- To control for the FWER at 0.05, we will use the Bonferroni correction.


In [4]:
counties = pd.read_csv('data/mh_analysis_ready.csv', index_col=0)
counties

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),Limited Life Expectancy (%),Plant primary fuel category_x,Plant annual net generation (MWh)
2,1001,AL,Autauga County,4,0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,NaN,NaN,NaN
3,1003,AL,Baldwin County,4,0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,NaN,NaN,NaN
4,1005,AL,Barbour County,4,0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,NaN,NaN,NaN
5,1007,AL,Bibb County,4,0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,NaN,NaN,NaN
6,1009,AL,Blount County,4,0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7805,51097,VA,King and Queen,3,1,287.0,28.000000,26.000000,8.000000,0.000000,1.000000,26.000000,4.000000,18.0,BIOMASS,"53,765"
7181,48347,TX,Nacogdoches,6,1,424.0,15.000000,40.000000,10.000000,0.000000,3.000000,20.000000,4.000000,4.0,BIOMASS,"249,859"
3670,16009,ID,Benewah,10,1,1477.0,37.000000,39.000000,13.000000,1.000000,4.000000,18.000000,5.000000,23.0,BIOMASS,"8,800"
192,1069,AL,Houston,4,1,107.0,29.000000,47.000000,21.000000,0.000000,12.000000,25.000000,4.000000,25.0,NaN,"14,758,529"


In [5]:
def difference_of_means(df, group_label, numerical_col, abs_dif):
    """
    Calculate difference in means, dependent on one or two sided test
        df: dataframe
        numerical_col (string): a numerical column name
        binary_col (string): a binary column name that will be shuffled
        i (integer): simmulation random state
        abs_dif (bool): whether it is a one sided or two sided alternative hypothesis
    """
    series = df.groupby('Shuffled Label').mean().loc[:, numerical_col]
    if abs_dif == True:
        return abs(series.iloc[1] - series.iloc[0])
    else:
        return series.iloc[1] - series.iloc[0]

In [6]:
def one_simulated_difference_of_means(df, numerical_col, binary_col, i, abs_dif):
    """
   The function computes a single simulation of an A/B permutation and the difference of means.
    inputs
        df: dataframe
        numerical_col (string): a numerical column name
        binary_col (string): a binary column name that will be shuffled
        i (integer): simmulation random state
        abs_dif (bool): whether it is a one sided or two sided alternative hypothesis
    """
    shuffled_df = df.copy()
    
    shuffled_labels = df.sample(replace=False, frac = 1, random_state = i)[binary_col]
    shuf = shuffled_labels.to_numpy()
    shuffled_df['Shuffled Label'] = shuf
    selected_shuf = shuffled_df.loc[:, (numerical_col, 'Shuffled Label')]
    
    return difference_of_means(selected_shuf, 'Shuffled Label', numerical_col, abs_dif)   

In [7]:
def avg_difference_in_means(df, numerical_col, abs_dif, binary_col= 'has_plant'):
    """
   The function computes the p-value for a test of the following hypothesis test:
        H0 : There is no difference in the average value of numerical_col between the two
            groups specified in binary_col.
        H1 : The average value of numerical_col is different for the two groups specified
            in binary_col
    inputs
        numerical_col: a numerical column name
        binary_col: a binary column name
    """
    selected = df.loc[:, (numerical_col, binary_col)]
    series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col]
    if abs_dif == True:
        observed_difference = abs(series_obs.iloc[1] - series_obs.iloc[0])
    else: 
        observed_difference = series_obs.iloc[1] - series_obs.iloc[0]
    differences = []

    repetitions = 2500
    for i in np.arange(repetitions):
        new_difference = one_simulated_difference_of_means(df, numerical_col, binary_col, i, abs_dif)
        differences = np.append(differences, new_difference)                               

    empirical_p = np.count_nonzero(differences >= observed_difference) / repetitions #how many samples have as extreme of a difference?

    #print(f' observed dif: {observed_difference}, empirical p: {empirical_p}')
    return empirical_p
    

In [8]:
one_sided_cols = ['People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)', 'Limited English Speaking (%)']
two_sided_cols = ['Over Age 64 (%)', 'Under Age 5 (%)']
binary_col = ['has_plant']
pvals = {}

# one sided alternative hypothesis, namely higher % 
for i in one_sided_cols:
    pvals[f'{i} and {binary_col}'] = avg_difference_in_means(counties, i, abs_dif = True)
# two sided alternative hypothesis for ages 
for k in two_sided_cols:
    pvals[f'{k} and {binary_col}'] = avg_difference_in_means(counties, k, abs_dif = False)

pvals

{"People of Color (%) and ['has_plant']": 0.0,
 "Low Income (%) and ['has_plant']": 0.002,
 "Less Than High School Education (%) and ['has_plant']": 0.0092,
 "Unemployment Rate (%) and ['has_plant']": 0.4236,
 "Limited English Speaking (%) and ['has_plant']": 0.0,
 "Over Age 64 (%) and ['has_plant']": 0.0124,
 "Under Age 5 (%) and ['has_plant']": 1.0}

### Bonferroni Procedure and FWER Discoveries

In [9]:
fwer = 0.05
num_tests = len(one_sided_cols) + len(two_sided_cols)
fwer_threshold = fwer / num_tests
print(f'FWER threshold: {fwer_threshold}')
reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fwer_threshold]
#list(pvals.values())
reject_null

FWER threshold: 0.0071428571428571435


["People of Color (%) and ['has_plant']",
 "Low Income (%) and ['has_plant']",
 "Limited English Speaking (%) and ['has_plant']"]

### Benjamini–Yekutieli Procedure and FDR Discoveries 

In [10]:
p_sorted = sorted(list(pvals.values()))

m = len(p_sorted)  
k = np.arange(1, m+1)  # index of each test in sorted order
alpha = 0.05 # desired FPR
c_m = np.sum([1/i for i in range(1, m)]) # Benjamini–Yekutieli procedure's 'harmonic function' 
compare = (alpha * k) /(m * c_m) # B-y threshold
below_than = p_sorted <= compare
cols = {'k' : k, 'p-vals' : p_sorted, 'compare': compare, 'below than' : below_than}

ps = pd.DataFrame(cols)
ps


,k,p-vals,compare,below than
0,1,0.0000,0.002915,True
1,2,0.0000,0.005831,True
2,3,0.0020,0.008746,True
3,4,0.0092,0.011662,True
4,5,0.0124,0.014577,True
5,6,0.4236,0.017493,False
6,7,1.0000,0.020408,False


In [11]:
fdr_threshold = ps[ps['below than'] == True]['p-vals'].iloc[-1]
print(f'FDR threshold: {fdr_threshold}')
fdr_reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fdr_threshold]
#list(pvals.values())
fdr_reject_null

FDR threshold: 0.0124


["People of Color (%) and ['has_plant']",
 "Low Income (%) and ['has_plant']",
 "Less Than High School Education (%) and ['has_plant']",
 "Limited English Speaking (%) and ['has_plant']",
 "Over Age 64 (%) and ['has_plant']"]

### Specific Alternative Test and Controlling Power

Plants are disproportionately located in areas with higher % People of Color, specifically: **the difference in mean percentage of People of Color between counties with plants and counties without plants is 3%**. This is based on a similar study which found that communities with oil plants were 2.6% more Black, 4.2% more Latinx, and 1.4% more Asian than communities without oil plants. https://www.sciencedirect.com/science/article/pii/S221462962300110X#s0055 


In [26]:
df = counties
binary_col = 'has_plant'
numerical_col = 'People of Color (%)'
sd_err_of_mean_treatment = df[df[binary_col] == 1][numerical_col].std() / len(df[df[binary_col] == 1][numerical_col]) #treatments std/n
sd_err_of_mean_control = df[df[binary_col] == 0][numerical_col].std() / len(df[df[binary_col] == 0][numerical_col]) #control std/n
std = np.sqrt((sd_err_of_mean_treatment**2) + (sd_err_of_mean_control**2))
std

0.017547676049681434

In [27]:
from scipy.stats import norm
def calculate_power(df, numerical_col, binary_col= 'has_plant'):
    """
   The function computes the power for a neyman pearson test:
        H0 : There is no difference in the average value of numerical_col between the two
            groups specified in binary_col.
        H1 : The average value of numerical_col is 3%
    inputs
        numerical_col: a numerical column name
        binary_col: a binary column name
    """
    selected = df.loc[:, (numerical_col, binary_col)]
    series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col]
    observed_difference = series_obs.iloc[1] - series_obs.iloc[0]

    #calculate std of the difference between means
    #The standard deviation of a set of mean values is the standard error.
    sd_err_of_mean_treatment = (df[df[binary_col] == 1][numerical_col].std() **2) / len(df[df[binary_col] == 1][numerical_col]) #treatments std/n
    sd_err_of_mean_control = (df[df[binary_col] == 0][numerical_col].std() **2) / len(df[df[binary_col] == 0][numerical_col]) #control std/n
    std = np.sqrt((sd_err_of_mean_treatment) + (sd_err_of_mean_control))

    # Define the likelihood functions for the observed data according to the null and alternative hypothesis
    x_given_null = norm.pdf(observed_difference, 0, std)
    x_given_alt = norm.pdf(observed_difference, 3, std)


    #Identify a desired level of significance
    fpr = 0.05

    # likelihood ratio as test statistic
    #LR = x_given_alt/x_given_null # i actually dont think this is relevant?

    # threshold n -- 
    # threshold n = 95% quantile under the null ie area for potential FPRs (?)
    threshold = norm.ppf(1-fpr, loc=0, scale=std)
   
    # if LR > n ==> reject the null, if LR < n ==> fail to reject null

    #power = TPR : p( rejecting the null, given alt hypothesis is true) so p(LR > n) | x is alt
    # p( x > threshold) under the alternative
    power = 1 - norm.cdf(threshold, loc=3, scale=std)

    #The likelihood ratio is: 789.968085380066, and the power is: 0.99479786395962

    print(f' The likelihood ratio is: , the observed difference is {observed_difference}, the threshold is {threshold}, and the power is: {power}')
    if observed_difference > threshold:
        print('Because the observed difference is greater than the threshold, we reject the null')
    else:
        print('Because the observed difference is not greater than the threshold, we fail to reject the null')
    return power
    

In [28]:
calculate_power(counties, 'People of Color (%)')

 The likelihood ratio is: , the observed difference is 3.723997515463985, the threshold is 1.1729547454762077, and the power is: 0.99479786395962
Because the observed difference is greater than the threshold, we reject the null


0.99479786395962

### Results
- Summarize and interpret the results from the hypothesis tests themselves.
- 